## Upload of Results to local InfluxDB
This notebook is for testing the upload of the simulation results to InfluxDB and can be manually executed after the simulation to upload the results. Shoudl achieve the same results as upload_to_influxdb() in report.py.

In [1]:
import influxdb_client, os, time
from influxdb_client import InfluxDBClient, Point, WritePrecision
from influxdb_client.client.write_api import SYNCHRONOUS
import pandas as pd
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv('../../.env')

# Securely retrieve credentials from environment variables
token = os.environ.get("INFLUXDB_TOKEN")
org = os.environ.get("INFLUXDB_ORG", "EST")  # Default to "EST" if not set
url = os.environ.get("INFLUXDB_URL", "http://localhost:8086")  # Default URL

if not token:
    raise ValueError("INFLUXDB_TOKEN not found in .env file")

# print("Configuration loaded successfully from .env file")
# print(f"Token: {token}")
# print(f"Organization: {org}")
# print(f"URL: {url}")

client = influxdb_client.InfluxDBClient(url=url, token=token, org=org)

In [2]:
df = pd.read_csv("../../results/csv/simulation_data.csv", delimiter=',') # all the data
# traj_df = pd.read_csv("../../results/csv/trajectory_coords.csv", delimiter=',') # the trajectory data

# we artificially add a timestamp to the data to convert to datetime
now = datetime(2028, 5, 1, 9, 0, 0)
df['tofs'] = df['tofs'].apply(lambda x: now + timedelta(seconds=x))

# the full df
# df = pd.concat([df.reset_index(drop=True), traj_df.reset_index(drop=True)], axis=1)

In [3]:
# Write data to InfluxDB
bucket="NICE"
write_api = client.write_api(write_options=SYNCHRONOUS)
delete_api = client.delete_api()

# Delete all previous data from bucket
start = "1970-01-01T00:00:00Z"
stop =  datetime(2070, 1, 1, 0, 0, 0)
delete_api.delete(start, stop, '', bucket=bucket, org=org)



# the subsystems' times are not included because for now they are the same for all
# the .tag are used to index the data, and can be used for filtering


# Pre-build list of points
points = [
    Point("satellite_data")
        .tag("mode", int(row["modes"]))
        .tag("visible", int(row["vis"]))
        .field("visibility", float(row["vis"]))
        .field("data", float(row["storage"]))
        .field("data_payload", float(row["storage_payload"]))
        .field("data_HK", float(row["storage_HK"]))
        .field("battery", float(row["battery"]))
        .field("consumption", float(row["consumption"]))
        .field("generation", float(row["generation"]))
        .field("eclipse", float(row["eclipse"]))
        .field("modes", float(row["modes"]))
        .field("altitude", float(row["altitudes"]))
        .field("RAAN", float(row["RAANs"]))
        .field("AOP", float(row["AOPs"]))
        .field("ECC", float(row["ECCs"]))
        .field("INC", float(row["INCs"]))
        .field("density", float(row["density_array"]))
        .field("Lat", float(row["latitude_deg"]))
        .field("Lng", float(row["longitude_deg"]))
        .field("solar_cells_efficiency", float(row["solar_cells_efficiency"]))
        .time(row["tofs"], write_precision=WritePrecision.NS)
    for _, row in df.iterrows()
]

# Send all points at once
write_api.write(bucket=bucket, org=org, record=points)
print("Upload complete.")

Upload complete.
